In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings 
from scipy.stats import ttest_ind
warnings.filterwarnings("ignore")

In [0]:
# Read the table using Spark and convert to pandas
spark_df = spark.read.table("vendor_performance.gold.vendor_sales_summary")
df = spark_df.toPandas()
df.head()

## EDA on Gold Layer

In [0]:
df.describe().T


In [0]:
numerical_cols = df.select_dtypes(include=np.number).columns
plt.figure(figsize=(15,10))
for i, col in enumerate(numerical_cols):
    plt.subplot(4, 4, i+1)
    sns.histplot(df[col], kde=True, bins=30)
    plt.title(col)
plt.tight_layout()
plt.show()

In [0]:
numerical_cols = df.select_dtypes(include=np.number).columns
plt.figure(figsize=(15,10))
for i, col in enumerate(numerical_cols):
    plt.subplot(4, 4, i+1)
    sns.boxplot(y=df[col])
    plt.title(col)
plt.tight_layout()
plt.show()

![image_1770594818823.png](./image_1770594818823.png "image_1770594818823.png")

In [0]:
# Filtering out the anomalies
spark_df = spark.sql("""
                        select 
                          * 
                        from vendor_performance.gold.vendor_sales_summary
                        where 
                          GrossProfit > 0 and 
                          GrossProfitMargin > 0 and 
                          TotalSalesQuantity > 0
                    """)
df = spark_df.toPandas()
# df

In [0]:
numerical_cols = df.select_dtypes(include=np.number).columns
plt.figure(figsize=(15,10))
for i, col in enumerate(numerical_cols):
    plt.subplot(4, 4, i+1)
    sns.histplot(df[col], kde=True, bins=30)
    plt.title(col)
plt.tight_layout()
plt.show()

### Check within our data, top vendors and top brands

In [0]:
categorial_cols = ['VendorName', 'Description']

plt.figure(figsize=(12,5))
for i, col in enumerate(categorial_cols):
    plt.subplot(1, 2, i+1)
    sns.countplot(y=df[col], order=df[col].value_counts().index[:10]) # top 10 categories
    plt.title(f"Count Plot of {col}")
plt.tight_layout()
plt.show()

### Check relationship with numerical variables

In [0]:
plt.figure(figsize=(12,8))
correlation_matrix = df[numerical_cols].corr()
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', linewidth=0.5)
plt.title('Correlation Heatmap')
plt.show()

![image_1770595884061.png](./image_1770595884061.png "image_1770595884061.png")

### Question 1 - Identify Brands that need promotional or Pricing adjustments which exhibit lower sales performance but higher profit margins

In [0]:
# Fix: Use pandas groupby and aggregation
brand_performance = df.groupby('Description').agg({
    'TotalSalesDollars': 'sum',
    'GrossProfitMargin': 'mean'
}).reset_index()


In [0]:
low_sales_threshold = brand_performance['TotalSalesDollars'].quantile(0.15)
high_margin_threshold = brand_performance['GrossProfitMargin'].quantile(0.85)

In [0]:
target_brands = brand_performance[
    (brand_performance['TotalSalesDollars'] <= low_sales_threshold) &
    (brand_performance['GrossProfitMargin'] >= high_margin_threshold)
]
print("Brands with Low Sales but High Profit Margins")
display(target_brands.sort_values('TotalSalesDollars'))

In [0]:
plt.figure(figsize=(10,8))
sns.scatterplot(data=brand_performance,x='TotalSalesDollars', y='GrossProfitMargin', color='blue', label='All Brands', alpha=0.2)
sns.scatterplot(data=target_brands,x='TotalSalesDollars', y='GrossProfitMargin', color='red', label='Targe Brands')

plt.axhline(high_margin_threshold, color='black', linestyle='--', label='High Margin Threshold')
plt.axvline(low_sales_threshold, color='black', linestyle='--', label='Low Sales Threshold')

plt.title('Brand for Promotional or Pricing Adjustments')
plt.xlabel("Total Sales ($)")
plt.ylabel("Profit Margin (%)")
plt.legend()
plt.grid(True)
plt.show()

### Question 2 - Which vendors and brands demonstrate the highest sales performance? 

In [0]:
def format_dollars(value):
    if value >= 1_000_000:
        return f"{value / 1_000_000:.2f}M"
    elif value >= 1_000:
        return f"{value / 1_000:.2f}K"
    else:
        return str(value)

In [0]:
# Fix: Use pandas groupby and aggregation
# Top vendors by sales

# Top vendors
top_vendors = df.groupby('VendorName')['TotalSalesDollars'].sum().nlargest(10)
# Top brands
top_brands = df.groupby('Description')['TotalSalesDollars'].sum().nlargest(10)
top_brands.apply(lambda x: format_dollars(x))

In [0]:
plt.figure(figsize=(15,5))

#plot top vendors
plt.subplot(1, 2, 1)
ax1 = sns.barplot(y= top_vendors.index, x= top_vendors.values, palette='Blues_r')
plt.title('Top 10 Vendors by Sales')

for bar in ax1.patches:
    ax1.text(bar.get_width() + (bar.get_width() * 0.02),
             bar.get_y() + bar.get_height()/2,
             format_dollars(bar.get_width()),
             ha='left', va='center', fontsize=10, color='black')

#plot top brands 
plt.subplot(1, 2, 2)
ax2 = sns.barplot(y= top_brands.index.astype(str), x= top_brands.values, palette='Reds_r')
plt.title('Top 10 Brands by Sales')

for bar in ax2.patches:
    ax2.text(bar.get_width() + (bar.get_width() * 0.02),
             bar.get_y() + bar.get_height()/2,
             format_dollars(bar.get_width()),
             ha='left', va='center', fontsize=10, color='black')

plt.tight_layout()
plt.show()

### Question 3 - Which vendors contribute the most to total purchase dollars? 

In [0]:
vendor_performance = df.groupby('VendorName').agg({
    'TotalPurchaseDollars': 'sum',
    'GrossProfit': 'sum',
    'TotalSalesDollars' : 'sum'
}).reset_index()

In [0]:
vendor_performance['PurchaseContribution%'] = vendor_performance['TotalPurchaseDollars'] / vendor_performance['TotalPurchaseDollars'].sum()

In [0]:
vendor_performance = round(vendor_performance.sort_values('PurchaseContribution%' , ascending = False),2)

In [0]:
#display top 10 vendors
top_vendors = vendor_performance.head(10)
top_vendors['TotalSalesDollars'] = top_vendors['TotalSalesDollars'].apply(format_dollars)
top_vendors['TotalPurchaseDollars'] = top_vendors['TotalPurchaseDollars'].apply(format_dollars)
top_vendors['GrossProfit'] = top_vendors['GrossProfit'].apply(format_dollars)
top_vendors

In [0]:
top_vendors['Cumulative_Contributon%'] = top_vendors['PurchaseContribution%'].cumsum()
# top_vendors

In [0]:
fig, ax1 = plt.subplots(figsize=(10,6))

#bar plot for purchase contribution % 
sns.barplot(x=top_vendors['VendorName'], y=top_vendors['PurchaseContribution%'], palette="mako", ax=ax1)

for i, value in enumerate(top_vendors['PurchaseContribution%']):
    ax1.text(i, value - 1, str(value)+'%', ha="center", fontsize=10, color="white")

ax2 = ax1.twinx()
ax2.plot(top_vendors['VendorName'], top_vendors['Cumulative_Contributon%'], color='red', marker='o', linestyle='dashed', label='Cumulative')

ax1.set_xticklabels(top_vendors['VendorName'], rotation=90)
ax1.set_ylabel('Purchase Contribution %', color='blue')
ax2.set_ylabel('Cumulative Contribution %', color='red')
ax1.set_xlabel('Vendors')
ax1.set_title('Pareto Chart: Vendor Contribution to Total Purchases')

# Horizontal line at 100% for reference
ax2.axhline(y=100, color='gray', linestyle='dashed', alpha=0.7)
ax2.legend(loc='upper right')

plt.show()

### Question 4 - How much of total procurement is dependent on top vendors?

In [0]:
print(f"{round(top_vendors['PurchaseContribution%'].sum(), 2)}")

In [0]:
# Extracting data from the top_vendors DataFrame
vendors = list(top_vendors['VendorName'].values)
purchase_contributions = list(top_vendors['PurchaseContribution%'].values)
total_contribution = sum(purchase_contributions)
remaining_contribution = 1 - total_contribution

# Append "Other Vendors" category to capture the remaining percentage
vendors.append("Other Vendors")
purchase_contributions.append(remaining_contribution)

# --- Donut Chart ---
fig, ax = plt.subplots(figsize=(8, 8))
wedges, texts, autotexts = ax.pie(purchase_contributions, labels=vendors, autopct='%1.1f%%',
                                  startangle=140, pctdistance=0.85, colors=plt.cm.Paired.colors)

# Draw a white circle in the center to create a "donut" effect
centre_circle = plt.Circle((0, 0), 0.70, fc='white')
fig.gca().add_artist(centre_circle)

total_contribution = total_contribution * 100
# Add Total Contribution annotation in the center of the donut
plt.text(0, 0, f"Top 10 Total:\n{total_contribution:.2f}%", 
         fontsize=14, fontweight='bold', ha='center', va='center')

plt.title("Top 10 Vendor's Purchase Contribution (%)")
plt.show()

### Question 5 - Does purchasing in bulk reduce the unit proce, and what is the optimal purchase volume for cost savings? 

In [0]:
# Calculate Unit Purchase Price
df['UnitPurchasePrice'] = df['TotalPurchaseDollars'] / df['TotalPurchaseQuantity']

# Binning Order Size into Quantiles
df["OrderSize"] = pd.qcut(df["TotalPurchaseQuantity"], q=3, labels=["Small", "Medium", "Large"])

In [0]:
# Fix: Use pandas groupby and mean aggregation
order_size_mean = df.groupby('OrderSize')['UnitPurchasePrice'].mean()
order_size_mean

In [0]:
plt.figure(figsize=(10, 6))
sns.boxplot(data=df, x="OrderSize", y="UnitPurchasePrice", palette="Set2")
plt.title("Impact of Bulk Purchasing on Unit Price")
plt.xlabel("Order Size")
plt.ylabel("Average Unit Purchase Price")
plt.show()

### Question 6 - Which vendors have low inventory turnover, indicating excess stock and slow-moving products? 

In [0]:
df[df['StockTurnover'] < 1].groupby('VendorName')[['StockTurnover']].mean().sort_values('StockTurnover', ascending=True).head(10)

### Question 7 - How much capital is locked in unsold inventory per vendor, and which vendors contribute the most to it? 

In [0]:
df["UnsoldInventoryValue"] = (df["TotalPurchaseQuantity"] - df["TotalSalesQuantity"]) * df["PurchasePrice"]
print('Total Unsold Capital:', format_dollars(df["UnsoldInventoryValue"].sum()))

In [0]:
# Aggregate Capital Locked per Vendor
inventory_value_per_vendor = df.groupby("VendorName")["UnsoldInventoryValue"].sum().reset_index()

# Sort Vendors with the Highest Locked Capital
inventory_value_per_vendor = inventory_value_per_vendor.sort_values(by="UnsoldInventoryValue", ascending=False)
inventory_value_per_vendor['UnsoldInventoryValue'] = inventory_value_per_vendor['UnsoldInventoryValue'].apply(format_dollars)
inventory_value_per_vendor.head(10)

### Question 8 - what is the 95% confidence intervals for profit margins of top-performing and low-performing vendors

In [0]:
df

In [0]:
top_threshold = df["TotalSalesDollars"].quantile(0.75)
low_threshold = df["TotalSalesDollars"].quantile(0.25)

top_vendors = df[df["TotalSalesDollars"] >= top_threshold]["GrossProfitMargin"].dropna()
low_vendors = df[df["TotalSalesDollars"] <= low_threshold]["GrossProfitMargin"].dropna()

In [0]:
from scipy import stats
def confidence_interval(data, confidence=0.95):
    mean_val = np.mean(data)
    std_err = np.std(data, ddof=1) / np.sqrt(len(data))  # Standard error
    t_critical = stats.t.ppf((1 + confidence) / 2, df=len(data) - 1)
    margin_of_error = t_critical * std_err
    return mean_val, mean_val - margin_of_error, mean_val + margin_of_error

In [0]:
top_mean, top_lower, top_upper = confidence_interval(top_vendors)
low_mean, low_lower, low_upper = confidence_interval(low_vendors)

print(f"Top Vendors 95% CI: ({top_lower:.2f}, {top_upper:.2f}), Mean: {top_mean:.2f}")
print(f"Low Vendors 95% CI: ({low_lower:.2f}, {low_upper:.2f}), Mean: {low_mean:.2f}")

plt.figure(figsize=(12, 6))

# Top Vendors Plot
sns.histplot(top_vendors, kde=True, color="blue", bins=30, alpha=0.5, label="Top Vendors")
plt.axvline(top_lower, color="blue", linestyle="--", label=f"Top Lower: {top_lower:.2f}")
plt.axvline(top_upper, color="blue", linestyle="--", label=f"Top Upper: {top_upper:.2f}")
plt.axvline(top_mean, color="blue", linestyle="-", label=f"Top Mean: {top_mean:.2f}")

# Low Vendors Plot
sns.histplot(low_vendors, kde=True, color="red", bins=30, alpha=0.5, label="Low Vendors")
plt.axvline(low_lower, color="red", linestyle="--", label=f"Low Lower: {low_lower:.2f}")
plt.axvline(low_upper, color="red", linestyle="--", label=f"Low Upper: {low_upper:.2f}")
plt.axvline(low_mean, color="red", linestyle="-", label=f"Low Mean: {low_mean:.2f}")

# Finalize Plot
plt.title("Confidence Interval Comparison: Top vs. Low Vendors (Profit Margin)")
plt.xlabel("Profit Margin (%)")
plt.ylabel("Frequency")
plt.legend()
plt.grid(True)
plt.show()

![image_1770606788230.png](./image_1770606788230.png "image_1770606788230.png")

## Question 9 - Hypothesis testing 
![image_1770606906537.png](./image_1770606906537.png "image_1770606906537.png")

In [0]:
top_threshold = df["TotalSalesDollars"].quantile(0.75)
low_threshold = df["TotalSalesDollars"].quantile(0.25)

top_vendors = df[df["TotalSalesDollars"] >= top_threshold]["GrossProfitMargin"].dropna()
low_vendors = df[df["TotalSalesDollars"] <= low_threshold]["GrossProfitMargin"].dropna()

# Perform Two-Sample T-Test
t_stat, p_value = ttest_ind(top_vendors, low_vendors, equal_var=False)

# Print results
print(f"T-Statistic: {t_stat:.4f}, P-Value: {p_value:.4f}")
if p_value < 0.05:
    print("Reject H0: There is a significant difference in profit margins between top and low-performing vendors.")
else:
    print("Fail to Reject H0: No significant difference in profit margins.")